In [1]:
import numpy as np
import pandas as pd
import sys

from pathlib import Path
from typing import Tuple

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.vis as vis

from config_handlers.eval_handler import EvalHandler
from config_handlers.rk_handler import RKHandler

In [2]:
rk_hdl = RKHandler()
eval_hdl = EvalHandler()

model_list = rk_hdl.get_model_list()
rk_hdl.set_curr_model(model_list[0])
ds_list = rk_hdl.get_ds_list()

all_ds = ds_list * 4
all_models = model_list * 4
all_models.sort()

MC_DATASETS = ["seed", "vqav2"]
OPEN_DATASETS = ["llava-bench", "mmbench"]

CHARTS_DIR = Path("failure_charts")
data_io.make_dir(CHARTS_DIR)

[Errno 17] File exists: 'failure_charts'


In [3]:
def load_data(version: str = "det") -> Tuple[dict, dict]:
    samples = {m: {ds: {} for ds in ds_list} for m in model_list}
    summaries = {m: {ds: {} for ds in ds_list} for m in model_list}

    for model, ds in zip(all_models, all_ds):
        if version == "sam":
            base_fp = Path("0_stats", "similarity_nucleus")
        else:
            base_fp = Path("0_stats", "similarity")

        samples[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "samples.json")
        )
        summaries[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "summary.json")
        )

    return samples, summaries

In [4]:
def load_eval_responses():
    data = {m: {ds: {}} for m in model_list for ds in ds_list}
    for model, ds in zip(all_models, all_ds):
        eval_hdl.set_curr_model(model)
        eval_hdl.set_curr_ds(ds)
        data[model][ds] = data_io.load_json(Path("..", eval_hdl.get_eval_path()))
    return data

In [5]:
samples, summaries = load_data()
eval_data = load_eval_responses()

### Accuracy of models

In [6]:
eval_data_small = {m: {ds: 0.0 for ds in eval_data[m]} for m in eval_data}
for m in eval_data:
    print(f">> Eval for {m.upper()}")
    for ds in eval_data[m]:
        if "f1" in eval_data[m][ds]["measures"]:
            print(f" - F1 on {ds}: {eval_data[m][ds]['measures']['f1']:.3f}")
        elif "accuracy" in eval_data[m][ds]["measures"]:
            print(f" - MC acc. on {ds}: {eval_data[m][ds]['measures']['accuracy']:.3f}")

>> Eval for INTERNVL2
 - MC acc. on vqav2: 0.660
 - F1 on llava-bench: 0.673
 - F1 on mmbench: 0.796
 - MC acc. on seed: 0.787
>> Eval for LLAVA-1.6
 - MC acc. on vqav2: 0.587
 - F1 on llava-bench: 0.669
 - F1 on mmbench: 0.793
 - MC acc. on seed: 0.687
>> Eval for MINIGPT4
 - MC acc. on vqav2: 0.260
 - F1 on llava-bench: 0.634
 - F1 on mmbench: 0.632
 - MC acc. on seed: 0.127
>> Eval for SHAREGPT4V
 - MC acc. on vqav2: 0.640
 - F1 on llava-bench: 0.690
 - F1 on mmbench: 0.706
 - MC acc. on seed: 0.660


### Explore values
> All good here

In [7]:
for m, ds in zip(all_models, all_ds):
    print(f">> Results for {m.upper()} on {ds.upper()}")
    print(f"- Num. entries: {len(samples[m][ds])}")
    print(f"- Avg. SIM: {summaries[m][ds]['cosine']:.3f}")
    # print(f"- Std. SIM: {summaries[m][ds]['std_sim']}")
    print("-" * 30)

>> Results for INTERNVL2 on LLAVA-BENCH
- Num. entries: 58
- Avg. SIM: 0.565
------------------------------
>> Results for INTERNVL2 on MMBENCH
- Num. entries: 139
- Avg. SIM: 0.598
------------------------------
>> Results for INTERNVL2 on SEED
- Num. entries: 150
- Avg. SIM: 0.544
------------------------------
>> Results for INTERNVL2 on VQAV2
- Num. entries: 150
- Avg. SIM: 0.601
------------------------------
>> Results for LLAVA-1.6 on LLAVA-BENCH
- Num. entries: 44
- Avg. SIM: 0.520
------------------------------
>> Results for LLAVA-1.6 on MMBENCH
- Num. entries: 133
- Avg. SIM: 0.573
------------------------------
>> Results for LLAVA-1.6 on SEED
- Num. entries: 129
- Avg. SIM: 0.534
------------------------------
>> Results for LLAVA-1.6 on VQAV2
- Num. entries: 142
- Avg. SIM: 0.548
------------------------------
>> Results for MINIGPT4 on LLAVA-BENCH
- Num. entries: 54
- Avg. SIM: 0.485
------------------------------
>> Results for MINIGPT4 on MMBENCH
- Num. entries: 102
- 

Plotting Similarity data

In [8]:
vis.plot_similarity(
    samples, f"{CHARTS_DIR}/distrib_similarity.pdf", measure_key="cosine"
)

#### Create ranges wrt cosine

Compute quartiles, IQR, and min/max

In [9]:
samples["internvl2"]["llava-bench"].values()

dict_values([{'ged': 32.0, 'iou': {'iou_nodes': 0.08695652173913043, 'iou_edges': 0.0}, 'cosine': 0.6553099751472473}, {'ged': 35.0, 'iou': {'iou_nodes': 0.08, 'iou_edges': 0.0}, 'cosine': 0.7163957357406616}, {'ged': 25.0, 'iou': {'iou_nodes': 0.05555555555555555, 'iou_edges': 0.0}, 'cosine': 0.5880377292633057}, {'ged': 22.0, 'iou': {'iou_nodes': 0.0, 'iou_edges': 0.0}, 'cosine': 0.4579523503780365}, {'ged': 16.0, 'iou': {'iou_nodes': 0.0, 'iou_edges': 0.0}, 'cosine': 0.4237445592880249}, {'ged': 16.0, 'iou': {'iou_nodes': 0.16666666666666666, 'iou_edges': 0.0}, 'cosine': 0.6617593765258789}, {'ged': 23.0, 'iou': {'iou_nodes': 0.0, 'iou_edges': 0.0}, 'cosine': 0.38589805364608765}, {'ged': 19.0, 'iou': {'iou_nodes': 0.0, 'iou_edges': 0.0}, 'cosine': 0.41042956709861755}, {'ged': 30.0, 'iou': {'iou_nodes': 0.05, 'iou_edges': 0.07142857142857142}, 'cosine': 0.5574652552604675}, {'ged': 20.0, 'iou': {'iou_nodes': 0.058823529411764705, 'iou_edges': 0.0}, 'cosine': 0.6468072533607483}, {'

In [10]:
for m, ds in zip(all_models, all_ds):
    sample_cosine = [v["cosine"] for v in samples[m][ds].values()]

    summaries[m][ds]["q1"] = np.percentile(sample_cosine, 25)
    summaries[m][ds]["q2"] = np.percentile(sample_cosine, 50)
    summaries[m][ds]["q3"] = np.percentile(sample_cosine, 75)

    iqr = summaries[m][ds]["q3"] - summaries[m][ds]["q1"]
    summaries[m][ds]["bottom"] = summaries[m][ds]["q1"] - 1.5 * iqr
    summaries[m][ds]["top"] = summaries[m][ds]["q3"] + 1.5 * iqr
    summaries[m][ds]["max"] = np.max(sample_cosine)
    summaries[m][ds]["min"] = np.min(sample_cosine)

In [11]:
df = pd.DataFrame()
for m, ds in zip(all_models, all_ds):
    for id_q in samples[m][ds]:
        row = {
            "question_id": int(id_q),
            "model": m,
            "dataset": ds,
            "similarity": [samples[m][ds][id_q]["cosine"]],
        }
        temp_df = pd.DataFrame.from_dict(row)
        df = pd.concat([df, temp_df], ignore_index=True)

In [12]:
df.head()

,question_id,model,dataset,similarity
0,0,internvl2,llava-bench,0.655310
1,1,internvl2,llava-bench,0.716396
2,2,internvl2,llava-bench,0.588038
3,3,internvl2,llava-bench,0.457952
4,4,internvl2,llava-bench,0.423745


Check again the dataframe

> All good here

In [13]:
# for m, ds in zip(ALL_MODELS, ALL_DATASETS):
#     print('-'*30)
#     print(f'>> Results for {m.upper()} on {ds.upper()}')
#     print(f'- Num. entries: {len(df[(df["model"]==m) & (df["dataset"]==ds)])}')

In [14]:
df_dict = {m: {ds: pd.DataFrame() for ds in ds_list} for m in model_list}
for m, ds in zip(all_models, all_ds):
    df_dict[m][ds] = df[(df["model"] == m) & (df["dataset"] == ds)]

In [15]:
# for m, ds in zip(ALL_MODELS, ALL_DATASETS):
#     print(f'{m} + {ds}: {len(df_dict[m][ds])}')

In [16]:
for m, ds in zip(all_models, all_ds):
    quartiles = [
        0,
        summaries[m][ds]["q1"],
        summaries[m][ds]["q2"],
        summaries[m][ds]["q3"],
        float("inf"),
    ]

    df_dict[m][ds]["quartile"] = pd.cut(
        df_dict[m][ds]["similarity"], bins=quartiles, labels=["1", "2", "3", "4"]
    )
    df_dict[m][ds]["quartile_range"] = pd.qcut(df_dict[m][ds]["similarity"], 4)

/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/1350319147.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds]["quartile"] = pd.cut(
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/1350319147.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds]["quartile_range"] = pd.qcut(df_dict[m][ds]["similarity"], 4)
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/1350319147.py:10: SettingWithCopyWarning: 
A value is trying to be set on a 

> Checking quartile splits for each model+dataset. All good.

In [17]:
# for m, ds in zip(ALL_MODELS, ALL_DATASETS):
#     print(f'{m} + {ds}')
#     display(df_dict[m][ds].groupby(['quartile'], as_index=False).count())

> The splits below add up to 1679 which is the total number of RKs collected. Correct.

Using min-max values to create 3-way even splits for each model+dataset.

In our case, we have all values in the [0, 1] range. So, that's the one we split up.

In [18]:
all_values = []

for m, ds in zip(all_models, all_ds):
    all_values.extend(df_dict[m][ds]["similarity"])

# Get general min and max
min = np.min(all_values)
max = np.max(all_values)
avg = np.mean(all_values)
std = np.std(all_values)

print(f"Minimum cos_sim: {round(min, 3)}")
print(f"Maximum cos_sim: {round(max, 3)}")
print(f"Avg. cos_sim: {round(avg, 3)}")
print(f"Std. cos_sim: {round(std, 3)}")

# Compute the threshold to obtain
# - [min, th1]
# - [th1, th2]
# - [th2, max]
interval_size = (max - min) / 3
cosine_ranges = {
    idx: {"start": float(val), "end": float(val + interval_size)}
    for idx, val in enumerate(np.arange(min, max, interval_size))
}

print(cosine_ranges)

Minimum cos_sim: -0.01
Maximum cos_sim: 0.936
Avg. cos_sim: 0.55
Std. cos_sim: 0.166
{0: {'start': -0.00975567102432251, 'end': 0.3056259751319885}, 1: {'start': 0.3056259751319885, 'end': 0.6210076212882996}, 2: {'start': 0.6210076212882996, 'end': 0.9363892674446106}}


**Add to the dataframes information about right/wrong/accuracy answer**

Create new column with eval outputs (to use later for measures and vis)
> The values below are correct, they are based on eval files with all responses

In [19]:
new_col = {model: {ds: {} for ds in eval_data[model]} for model in eval_data}

for model, ds in zip(all_models, all_ds):
    # Flatten to list of IDs
    print(f"{model.upper()} + {ds.upper()}")
    for k in eval_data[model][ds]:
        if k != "measures":
            if ds in OPEN_DATASETS and k == "ranges":
                ids_flat = []
                for r in eval_data[model][ds][k]:
                    ids_flat.extend(
                        [q["question_id"] for q in eval_data[model][ds][k][r]]
                    )
                    new_col[model][ds].update({int(idx): r for idx in ids_flat})
            else:
                ids_flat = [q["question_id"] for q in eval_data[model][ds][k]]
                new_col[model][ds].update({int(idx): k for idx in ids_flat})

        if ds in OPEN_DATASETS and k != "measures":
            if k == "invalid_resp":
                ids_flat = [q["question_id"] for q in eval_data[model][ds][k]]

    print(f"Found {len(new_col[model][ds].keys())} for {model} + {ds}.")


# fix naming of llava column after processing it
# for m in eval_data:
#     new_col[m]["llava"] = new_col[m].pop("llava-bench")

INTERNVL2 + LLAVA-BENCH
Found 60 for internvl2 + llava-bench.
INTERNVL2 + MMBENCH
Found 139 for internvl2 + mmbench.
INTERNVL2 + SEED
Found 150 for internvl2 + seed.
INTERNVL2 + VQAV2
Found 150 for internvl2 + vqav2.
LLAVA-1.6 + LLAVA-BENCH
Found 60 for llava-1.6 + llava-bench.
LLAVA-1.6 + MMBENCH
Found 139 for llava-1.6 + mmbench.
LLAVA-1.6 + SEED
Found 150 for llava-1.6 + seed.
LLAVA-1.6 + VQAV2
Found 150 for llava-1.6 + vqav2.
MINIGPT4 + LLAVA-BENCH
Found 60 for minigpt4 + llava-bench.
MINIGPT4 + MMBENCH
Found 139 for minigpt4 + mmbench.
MINIGPT4 + SEED
Found 150 for minigpt4 + seed.
MINIGPT4 + VQAV2
Found 150 for minigpt4 + vqav2.
SHAREGPT4V + LLAVA-BENCH
Found 60 for sharegpt4v + llava-bench.
SHAREGPT4V + MMBENCH
Found 139 for sharegpt4v + mmbench.
SHAREGPT4V + SEED
Found 150 for sharegpt4v + seed.
SHAREGPT4V + VQAV2
Found 150 for sharegpt4v + vqav2.


In [20]:
def map_func(row, new_col):
    try:
        return new_col[row["model"]][row["dataset"]][row["question_id"]]
    except Exception as e:
        print(e)
        return None


for m, ds in zip(all_models, all_ds):
    df_dict[m][ds]["outcome"] = df_dict[m][ds].apply(
        lambda x: map_func(x, new_col), axis=1
    )

/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/1271795053.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds]["outcome"] = df_dict[m][ds].apply(
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/1271795053.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds]["outcome"] = df_dict[m][ds].apply(
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/1271795053.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

In [21]:
df_dict["internvl2"]["llava-bench"].head()

,question_id,model,dataset,similarity,quartile,quartile_range,outcome
0,0,internvl2,llava-bench,0.655310,3,"(0.579, 0.656]",80_100
1,1,internvl2,llava-bench,0.716396,4,"(0.656, 0.829]",80_100
2,2,internvl2,llava-bench,0.588038,3,"(0.579, 0.656]",80_100
3,3,internvl2,llava-bench,0.457952,1,"(0.256, 0.473]",80_100
4,4,internvl2,llava-bench,0.423745,1,"(0.256, 0.473]",80_100


> Looking at model x dataset combo with the cell below -> everything is in order

In [22]:
# Warning: big print here

# for m in df['model'].unique():
#     for ds in df['dataset'].unique():
#         with pd.option_context('display.max_rows', None):
#             print(f'>> Results for {m.upper()} on {ds.upper()}')
#             display(df[(df['model']==m) & (df['dataset']==ds)])

In [23]:
for m, ds in zip(all_models, all_ds):
    # Add column for question_type
    df_dict[m][ds]["question_type"] = df_dict[m][ds].apply(
        lambda x: "open" if x["dataset"] in OPEN_DATASETS else "closed", axis=1
    )
    # Add split outcome column depending on question type
    df_dict[m][ds]["closed_res"] = df_dict[m][ds].apply(
        lambda x: x["outcome"] if x["question_type"] == "closed" else None, axis=1
    )
    df_dict[m][ds]["open_res"] = df_dict[m][ds].apply(
        lambda x: x["outcome"] if x["question_type"] == "open" else None, axis=1
    )

df_dict["internvl2"]["llava-bench"].head()

/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/405406570.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds]["question_type"] = df_dict[m][ds].apply(
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/405406570.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds]["closed_res"] = df_dict[m][ds].apply(
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/405406570.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

,question_id,model,dataset,similarity,quartile,quartile_range,outcome,question_type,closed_res,open_res
0,0,internvl2,llava-bench,0.655310,3,"(0.579, 0.656]",80_100,open,None,80_100
1,1,internvl2,llava-bench,0.716396,4,"(0.656, 0.829]",80_100,open,None,80_100
2,2,internvl2,llava-bench,0.588038,3,"(0.579, 0.656]",80_100,open,None,80_100
3,3,internvl2,llava-bench,0.457952,1,"(0.256, 0.473]",80_100,open,None,80_100
4,4,internvl2,llava-bench,0.423745,1,"(0.256, 0.473]",80_100,open,None,80_100


In [24]:
# for ds in df['dataset'].unique():
#     display(df[df['dataset']==ds].head())

> All data is in order up to this point.

Fix dataframe columns

In [25]:
# Format 'closed_res' and 'open_res' columns
def fix_closed_res(row):
    if row["question_type"] == "closed":
        return "correct" if row["closed_res"] == "correct_res" else "wrong"
    else:
        return None


for m, ds in zip(all_models, all_ds):
    # Drop 'outcome' column, its job is done
    df_dict[m][ds].drop(["outcome"], axis=1, inplace=True)
    df_dict[m][ds]["closed_res"] = df_dict[m][ds].apply(
        lambda x: fix_closed_res(x), axis=1
    )

/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/884169553.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds].drop(["outcome"], axis=1, inplace=True)
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/884169553.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dict[m][ds]["closed_res"] = df_dict[m][ds].apply(
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_83640/884169553.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the document

In [26]:
df_dict["internvl2"]["llava-bench"].head()

,question_id,model,dataset,similarity,quartile,quartile_range,question_type,closed_res,open_res
0,0,internvl2,llava-bench,0.655310,3,"(0.579, 0.656]",open,None,80_100
1,1,internvl2,llava-bench,0.716396,4,"(0.656, 0.829]",open,None,80_100
2,2,internvl2,llava-bench,0.588038,3,"(0.579, 0.656]",open,None,80_100
3,3,internvl2,llava-bench,0.457952,1,"(0.256, 0.473]",open,None,80_100
4,4,internvl2,llava-bench,0.423745,1,"(0.256, 0.473]",open,None,80_100


> All good until here.

### Explore BERTScore values

In [27]:
# Prepare a bit the data
bert_scores = {model: {} for model in model_list}

for model in eval_data:
    for ds in OPEN_DATASETS:
        bert_scores[model][ds] = []
        for r in eval_data[model][ds]["ranges"]:
            bert_scores[model][ds].extend(eval_data[model][ds]["ranges"][r])

for model in bert_scores:
    for ds in bert_scores[model]:
        bert_scores[model][ds] = {
            question["question_id"]: question["measures"]["f1"]
            for question in bert_scores[model][ds]
        }
        print(f"Formatted {len(bert_scores[model][ds])} entries for {model} on {ds}.")

Formatted 60 entries for internvl2 on llava-bench.
Formatted 139 entries for internvl2 on mmbench.
Formatted 60 entries for llava-1.6 on llava-bench.
Formatted 139 entries for llava-1.6 on mmbench.
Formatted 60 entries for minigpt4 on llava-bench.
Formatted 139 entries for minigpt4 on mmbench.
Formatted 60 entries for sharegpt4v on llava-bench.
Formatted 139 entries for sharegpt4v on mmbench.


In [28]:
def plot_bar(data, model, dataset):
    width = 0.5
    fig, ax = plt.subplots()
    fig.set_figwidth(20)
    # q_ids = list(data.keys())
    q_ids = list(np.arange(len(data.keys())))
    scores = list(data.values())

    p = ax.bar(q_ids, scores, width)

    # Add lines and counts for each threshold
    for y in np.arange(0.2, 1, 0.2):
        count = len([s for s in scores if s >= y])
        ax.text(-2, y, count, transform=ax.get_xaxis_transform())
        ax.axhline(y=y, color="r", linestyle=":")

    count = len([s for s in scores if s >= 0.7])
    ax.text(-2, 0.7, count, transform=ax.get_xaxis_transform())
    ax.axhline(y=0.7, color="g", linestyle="--")

    ax.set_title(f"BERTScore for {model.upper()} on {dataset.upper()}")
    # ax.legend(loc='upper right')
    plt.show()


for model in bert_scores:
    for ds in bert_scores[model]:
        plot_bar(bert_scores[model][ds], model, ds)

NameError: name 'plt' is not defined

### Compute counts

Count based on cosine thresholds

In [29]:
# The following are the ranges based on the obtained value range
cosine_ranges

{0: {'start': -0.00975567102432251, 'end': 0.3056259751319885},
 1: {'start': 0.3056259751319885, 'end': 0.6210076212882996},
 2: {'start': 0.6210076212882996, 'end': 0.9363892674446106}}

In [30]:
list(enumerate(np.linspace(0, 1, 4)))

[(0, np.float64(0.0)),
 (1, np.float64(0.3333333333333333)),
 (2, np.float64(0.6666666666666666)),
 (3, np.float64(1.0))]

In [31]:
step = round(1 / 3, 3)
cosine_ranges_eq = {
    idx: {"start": float(round(val, 3)), "end": float(round(val + step, 3))}
    for idx, val in enumerate(np.linspace(0, 1, 4))
}

# remove last range, meaningless here
cosine_ranges_eq.pop(3)
cosine_ranges_eq

{0: {'start': 0.0, 'end': 0.333},
 1: {'start': 0.333, 'end': 0.666},
 2: {'start': 0.667, 'end': 1.0}}

In [32]:
counts_per_model = {m: {ds: {} for ds in ds_list} for m in model_list}

for m, ds in zip(all_models, all_ds):
    curr_df = df_dict[m][ds]
    for r in cosine_ranges_eq:
        counts_per_model[m][ds][r] = len(
            curr_df[
                (curr_df["similarity"] >= cosine_ranges_eq[r]["start"])
                & (curr_df["similarity"] <= cosine_ranges_eq[r]["end"])
            ]
        )

In [33]:
counts_per_type = {idx: {} for idx in cosine_ranges_eq}

for idx, r in cosine_ranges_eq.items():
    for m, ds in zip(all_models, all_ds):
        curr_df = df_dict[m][ds]
        if m not in counts_per_type[idx]:
            counts_per_type[idx][m] = {}
        if ds not in counts_per_type[idx][m]:
            counts_per_type[idx][m][ds] = 0.0

        counts_per_type[idx][m][ds] = len(
            curr_df[
                (curr_df["similarity"] >= r["start"])
                & (curr_df["similarity"] < r["end"])
            ]
        )

In [34]:
counts_per_type

{0: {'internvl2': {'llava-bench': 2, 'mmbench': 8, 'seed': 17, 'vqav2': 6},
  'llava-1.6': {'llava-bench': 8, 'mmbench': 12, 'seed': 18, 'vqav2': 14},
  'minigpt4': {'llava-bench': 12, 'mmbench': 17, 'seed': 20, 'vqav2': 5},
  'sharegpt4v': {'llava-bench': 6, 'mmbench': 10, 'seed': 30, 'vqav2': 1}},
 1: {'internvl2': {'llava-bench': 44, 'mmbench': 81, 'seed': 93, 'vqav2': 89},
  'llava-1.6': {'llava-bench': 25, 'mmbench': 81, 'seed': 82, 'vqav2': 100},
  'minigpt4': {'llava-bench': 31, 'mmbench': 59, 'seed': 97, 'vqav2': 78},
  'sharegpt4v': {'llava-bench': 33, 'mmbench': 80, 'seed': 61, 'vqav2': 15}},
 2: {'internvl2': {'llava-bench': 12, 'mmbench': 50, 'seed': 40, 'vqav2': 54},
  'llava-1.6': {'llava-bench': 11, 'mmbench': 40, 'seed': 29, 'vqav2': 28},
  'minigpt4': {'llava-bench': 11, 'mmbench': 25, 'seed': 24, 'vqav2': 38},
  'sharegpt4v': {'llava-bench': 10, 'mmbench': 44, 'seed': 18, 'vqav2': 5}}}

In [35]:
counts_plot = {idx: {} for idx in cosine_ranges_eq}

for idx, r in cosine_ranges_eq.items():
    for m, ds in zip(all_models, all_ds):
        curr_df = df_dict[m][ds]
        if ds not in counts_plot[idx]:
            counts_plot[idx][ds] = []

        counts_plot[idx][ds].append(
            len(
                curr_df[
                    (curr_df["similarity"] >= r["start"])
                    & (curr_df["similarity"] < r["end"])
                ]
            )
        )

# Aligning the data for plotting
# Swap: 0 (Type 3 failures) <-> 2 (Type 1 failures)
# Good: 1 (Type 2 failures)

temp = counts_plot[0]
counts_plot[0] = counts_plot[2]
counts_plot[2] = temp

counts_plot

{0: {'llava-bench': [12, 11, 11, 10],
  'mmbench': [50, 40, 25, 44],
  'seed': [40, 29, 24, 18],
  'vqav2': [54, 28, 38, 5]},
 1: {'llava-bench': [44, 25, 31, 33],
  'mmbench': [81, 81, 59, 80],
  'seed': [93, 82, 97, 61],
  'vqav2': [89, 100, 78, 15]},
 2: {'llava-bench': [2, 8, 12, 6],
  'mmbench': [8, 12, 17, 10],
  'seed': [17, 18, 20, 30],
  'vqav2': [6, 14, 5, 1]}}

In [36]:
counts_model = {m: {ds: len(samples[m][ds]) for ds in samples[m]} for m in samples}

counts_model

{'internvl2': {'llava-bench': 58, 'mmbench': 139, 'seed': 150, 'vqav2': 150},
 'llava-1.6': {'llava-bench': 44, 'mmbench': 133, 'seed': 129, 'vqav2': 142},
 'minigpt4': {'llava-bench': 54, 'mmbench': 102, 'seed': 141, 'vqav2': 123},
 'sharegpt4v': {'llava-bench': 49, 'mmbench': 135, 'seed': 109, 'vqav2': 21}}

Plotting

In [37]:
out_path = Path(CHARTS_DIR, "behaviour_counts.pdf")
vis.plot_behaviour_counts(counts_plot, out_path)

/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:486: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


# Old stuff from here

Compute counts per range and per model response.

In [ ]:
for m, ds in zip(all_models, all_ds):
    print(f">> {m} + {ds}")
    curr_df = df_dict[m][ds]
    for idx, r in cosine_ranges.items():
        small_df = curr_df[
            (curr_df["similarity"] >= r["start"]) & (curr_df["similarity"] <= r["end"])
        ]

        match idx:
            case 0:
                print("Type 3 failures")
            case 1:
                print("Type 2 failures")
            case 2:
                print("Type 1 failures")
            case _:
                print("Something went very wrong.")

        if ds in MC_DATASETS:
            groups = small_df.groupby(["closed_res"], as_index=False).count()
            # handle counts better
            correct_count = groups[groups["closed_res"] == "correct"]["question_id"]
            wrong_count = groups[groups["closed_res"] == "wrong"]["question_id"]

            if len(correct_count) > 0:
                correct_count = correct_count.item()
            else:
                correct_count = 0

            if len(wrong_count) > 0:
                wrong_count = wrong_count.item()
            else:
                wrong_count = 0

            print(f"- Correct answers: {correct_count}.")
            print(f"- Wrong answers: {wrong_count}.")
        else:
            # TODO: implement this
            pass

>> InternVL2 + llava
Type 3 failures
Type 2 failures
Type 1 failures
>> InternVL2 + mmbench
Type 3 failures
Type 2 failures
Type 1 failures
>> InternVL2 + seed
Type 3 failures
- Correct answers: 15.
- Wrong answers: 4.
Type 2 failures
- Correct answers: 81.
- Wrong answers: 24.
Type 1 failures
- Correct answers: 22.
- Wrong answers: 4.
>> InternVL2 + vqav2
Type 3 failures
- Correct answers: 2.
- Wrong answers: 2.
Type 2 failures
- Correct answers: 55.
- Wrong answers: 27.
Type 1 failures
- Correct answers: 42.
- Wrong answers: 22.
>> LLaVa-1.6 + llava
Type 3 failures
Type 2 failures
Type 1 failures
>> LLaVa-1.6 + mmbench
Type 3 failures
Type 2 failures
Type 1 failures
>> LLaVa-1.6 + seed
Type 3 failures
- Correct answers: 14.
- Wrong answers: 7.
Type 2 failures
- Correct answers: 58.
- Wrong answers: 29.
Type 1 failures
- Correct answers: 14.
- Wrong answers: 7.
>> LLaVa-1.6 + vqav2
Type 3 failures
- Correct answers: 3.
- Wrong answers: 9.
Type 2 failures
- Correct answers: 59.
- Wrong

Check against wrong/correct model responses

In [ ]:
q_labels = [str(val) for val in range(1, 5)]
quartile_dict = {
    m: {
        ds: {
            q: {"correct": [], "wrong": []}
            if ds in MC_DATASETS
            else {
                "p_0_20": [],
                "p_20_40": [],
                "p_40_60": [],
                "p_60_80": [],
                "p_80_100": [],
            }
            for q in q_labels
        }
        for ds in ds_list
    }
    for m in model_list
}

In [ ]:
for m in model_list:
    for ds in ds_list:
        # For now, check only MC questions
        if ds in MC_DATASETS:
            correct_ids = [q["question_id"] for q in eval_data[m][ds]["correct_res"]]
            wrong_ids = [q["question_id"] for q in eval_data[m][ds]["wrong_res"]]

            # Divide correct reponses
            for c_idx in correct_ids:
                q = df[
                    (df["id_question"] == c_idx)
                    & (df["dataset"] == ds)
                    & (df["model"] == m)
                ]["quartiles"].values
                if len(q) > 0:
                    q = q[0]
                    quartile_dict[m][ds][q]["correct"].append(c_idx)

            # Divide wrong responses
            for w_idx in wrong_ids:
                q = df[
                    (df["id_question"] == w_idx)
                    & (df["dataset"] == ds)
                    & (df["model"] == m)
                ]["quartiles"].values
                if len(q) > 0:
                    q = q[0]
                    quartile_dict[m][ds][q]["wrong"].append(w_idx)
        elif ds in OPEN_DATASETS:
            print(f"{m} -- {ds}")
            for k in eval_data[m][ds]:
                # Only consider accuracy buckets
                if k not in ["measures", "invalid_res"]:
                    curr_split_ids = [q["question_id"] for q in eval_data[m][ds][k]]
                    print(f"---- {k}: {curr_split_ids}")

                    for curr_idx in curr_split_ids:
                        q = df[
                            (df["id_question"] == curr_idx)
                            & (df["dataset"] == ds)
                            & (df["model"] == m)
                        ]["quartiles"].values
                        if len(q) > 0:
                            q = q[0]
                            print(f"Appending {curr_idx} to {m}-{ds}-{q}-{k}")
                            quartile_dict[m][ds][q][k].append(curr_idx)

with open("test.json", "w") as fp:
    json.dump(quartile_dict, fp, indent=4)

InternVL2 -- mmbench
---- p_0_20: []
---- p_20_40: []
---- p_40_60: [3001815, 3001793, 1000055, 2001793, 1001793, 2000055]
Appending 3001815 to InternVL2-mmbench-1-p_40_60
Appending 3001793 to InternVL2-mmbench-1-p_40_60
Appending 1000055 to InternVL2-mmbench-3-p_40_60
Appending 2001793 to InternVL2-mmbench-1-p_40_60
Appending 1001793 to InternVL2-mmbench-1-p_40_60
Appending 2000055 to InternVL2-mmbench-2-p_40_60
---- p_60_80: [2001830, 3001801, 1824, 2000078, 47, 2000070, 74, 3000074, 2000082, 3001809, 2001809, 2000053, 3000047, 1000049, 1795, 3000070, 2000062, 1001795, 3000078, 3000082, 1000069, 1830, 78, 2001824, 62, 2001792, 1000115, 1828, 2001828, 1001809, 2001825, 1801, 1800, 1000062, 1787, 2001800, 1000047, 3001830, 3000115, 38, 50, 97, 102, 114, 1000038, 1000050, 1000068, 1000089, 1000092, 1000094, 1000097, 1000102, 1000114, 2000038, 2000050, 2000068, 2000089, 2000092, 2000094, 2000097, 2000102, 2000114, 3000038, 3000050, 68, 3000068, 3000089, 3000092, 3000094, 3000097, 3000102

In [ ]:
eval_data["InternVL2"]["llava-bench"].keys()

dict_keys(['measures', 'p_0_20', 'p_20_40', 'p_40_60', 'p_60_80', 'p_80_100', 'invalid_res'])